# MZ세대 공무원 이직의도: 랜덤 포레스트와 SHAP

2025년 한국심리측정평가학회 춘계학술대회 구두 발표(SHAP 기반 설명 가능한 AI로 들여다본 공무원 이직의도의 심리적 메커니즘)에 쓴 분석입니다.

- 자료: 2023 공직생활실태조사 중 1980년 이후 출생 응답자. 예측변수 55개, 결과변수는 이직의도(있음/없음)
- 입력 파일: `data_cleaned.csv` (grad-coursework 저장소의 회귀분석 기말 프로젝트 `01_logistic_regression.Rmd`가 만드는 파일, 원자료는 공개하지 않음)
- 순서: PyCaret으로 여러 분류 모형 비교 → 랜덤 포레스트 하이퍼파라미터 탐색 → 시험 자료 성능 → 변수 중요도와 SHAP 해석
- 시험 자료 성능(재실행 확인): 정확도 .722, AUC .802

## 1. 준비

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from pycaret.classification import setup, compare_models, pull
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.model_selection import GridSearchCV, train_test_split

data = pd.read_csv("data_cleaned.csv")
data["Turnover_Intention"] = data["Turnover_Intention"].map({0: "No", 1: "Yes"})  # 결과변수: 이직의도
print(data.shape)
print(data["Turnover_Intention"].value_counts())

## 2. 분류 모형 비교 (PyCaret)

어떤 계열의 모형이 이 자료에 잘 맞는지 먼저 훑어봅니다. 10겹 층화 교차검증, 표준화, 학습 폴드 안에서 SMOTE로 소수 범주를 합성합니다.

In [ ]:
s = setup(
    data,
    target="Turnover_Intention",
    session_id=1,                     # 같은 결과가 나오도록 시드 고정
    fold=10,
    fold_strategy="stratifiedkfold",
    normalize=True,
    fix_imbalance=True,               # SMOTE
    n_jobs=-1,
)

best_model = compare_models(sort="AUC")   # 기본 분류 모형 전체를 AUC 순으로 비교
model_results = pull()
print(model_results)

In [ ]:
# AUC 상위 5개 모형의 지표 비교
top_models = model_results.sort_values(by="AUC", ascending=False).head(5)
top_models_melted = top_models.melt(id_vars=["Model"],
                                    value_vars=["AUC", "Accuracy", "Recall", "Prec.", "F1"],
                                    var_name="Metric", value_name="Score")

plt.figure(figsize=(12, 6))
sns.lineplot(data=top_models_melted, x="Metric", y="Score", hue="Model", marker="o")
plt.title("AUC 상위 5개 모형의 성능 지표")
plt.ylim(0.6, 0.85)
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 3. 랜덤 포레스트

해석에는 비교에서 상위권이었고 트리 기반 SHAP을 바로 쓸 수 있는 랜덤 포레스트를 썼습니다. 학습 70%, 시험 30%로 나누고 학습 자료 안에서 10겹 교차검증으로 하이퍼파라미터를 고릅니다. (이 단계는 SMOTE를 쓰지 않습니다.)

In [ ]:
X = data.drop("Turnover_Intention", axis=1)
y = data["Turnover_Intention"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

param_grid = {
    "n_estimators": [100, 300, 500, 800, 1000],
    "max_features": np.arange(2, int(np.sqrt(X_train.shape[1])) + 1, 2),  # 2, 4, 6
    "min_samples_leaf": [1, 3, 5, 7, 10],
    "random_state": [42],
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(),
    param_grid=param_grid,
    cv=10,
    scoring="roc_auc",
    n_jobs=-1,
)
grid_search.fit(X_train, y_train)

best_rf = grid_search.best_estimator_
print(grid_search.best_params_)   # 원 분석: n_estimators=800, max_features=6, min_samples_leaf=5

In [ ]:
# 시험 자료 성능
y_pred = best_rf.predict(X_test)
y_pred_proba = best_rf.predict_proba(X_test)[:, 1]   # "Yes"(이직의도 있음) 확률

print(f"정확도: {accuracy_score(y_test, y_pred):.4f}")
print(f"AUC: {roc_auc_score(y_test, y_pred_proba):.4f}")
print(classification_report(y_test, y_pred))

In [ ]:
# 불순도 기반 변수 중요도 (전체 변수)
vip = pd.Series(best_rf.feature_importances_, index=X_train.columns).sort_values()

plt.figure(figsize=(10, 14))
plt.barh(vip.index, vip.values, color="steelblue")
plt.title("랜덤 포레스트 변수 중요도")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

## 4. SHAP 해석

시험 자료에 대해 TreeExplainer로 SHAP 값을 구하고, "Yes"(이직의도 있음) 범주에 대한 값으로 중요도, 방향, 형태를 봅니다.

In [ ]:
explainer = shap.TreeExplainer(best_rf)
shap_values = explainer.shap_values(X_test)      # (사례, 변수, 범주)
shap_yes = shap_values[:, :, 1]                  # 두 번째 범주 "Yes"

shap.summary_plot(shap_yes, X_test, plot_type="bar")   # 평균 |SHAP| 중요도
shap.summary_plot(shap_yes, X_test)                    # 사례별 SHAP 분포와 방향

In [ ]:
# 평균 |SHAP|이 큰 상위 20개 변수의 의존성 그림
mean_abs_shap = pd.Series(np.abs(shap_yes).mean(axis=0), index=X_test.columns)
top_features = mean_abs_shap.sort_values(ascending=False).index[:20]

for feature in top_features:
    shap.dependence_plot(feature, shap_yes, X_test)